# RetailStream Inc. Data Engineering Pipeline

## Final Project - Celebal Technologies Internship

### Technologies Used
- PySpark
- Delta Lake
- Databricks
- Spark SQL
- Auto Loader

## Project Objective

The objective of this project is to build an end-to-end data pipeline for RetailStream Inc. using PySpark and Delta Lake. The pipeline processes historical batch data, incremental data, streaming transaction data, and late-arriving records while following the Bronze-Silver-Gold architecture.

In [0]:
# Import Libraries
from pyspark.sql import functions as F
from pyspark.sql.functions import *
from delta.tables import DeltaTable

In [0]:
dbutils.widgets.text("environment", "dev", "Environment")
dbutils.widgets.text("batch_file", "orders_2024_01.csv", "Batch File")

environment = dbutils.widgets.get("environment")
batch_file = dbutils.widgets.get("batch_file")

print("Environment :", environment)
print("Batch File :", batch_file)

Environment : dev
Batch File : orders_2024_01.csv


In [0]:
# Base Paths

BASE_PATH = "/Volumes/workspace/default/retailstream_data"

DATA_PATH = f"{BASE_PATH}/data"

BRONZE_PATH = f"{BASE_PATH}/delta/bronze"

SILVER_PATH = f"{BASE_PATH}/delta/silver"

GOLD_PATH = f"{BASE_PATH}/delta/gold"

CHECKPOINT_PATH = f"{BASE_PATH}/checkpoints"

In [0]:
# -----------------------------
# Helper Function
# -----------------------------

def read_csv(file_path):
    return (
        spark.read
            .option("header", True)
            .option("inferSchema", True)
            .csv(file_path)
    )

## Project Workflow

Landing Files

│

├── Historical Orders

├── Incremental Orders

├── Streaming Transactions

└── Late Arriving Orders

↓

Bronze Layer

Raw Delta Tables

↓

Silver Layer

Business Ready Clean Data

↓

Gold Layer

Aggregated Reports

↓

Power BI Dashboard

# Bronze Layer

The Bronze layer stores raw data exactly as received from multiple sources with only minimal transformations and audit columns.

## Task 1: 
### Load Historical Orders into Bronze Layer

This step reads the historical January order data from the landing zone, adds audit information, and stores it as a Delta table in the Bronze layer.

In [0]:
display(dbutils.fs.ls(f"{DATA_PATH}/batch_initial"))

path,name,size,modificationTime
dbfs:/Volumes/workspace/default/retailstream_data/data/batch_initial/orders_2024_01.csv,orders_2024_01.csv,1065,1783700282000


In [0]:
orders_df = read_csv(
    f"{DATA_PATH}/batch_initial/{batch_file}"
)

In [0]:
display(orders_df)

order_id,customer_id,product_id,quantity,unit_price,order_date,store_id,status
ORD001,C026,P008,1,28000,2024-01-11,S02,DELIVERED
ORD002,C021,P004,4,18000,2024-01-12,S03,DELIVERED
ORD003,C020,P003,2,50000,2024-01-04,S01,DELIVERED
ORD004,C031,P007,1,18000,2024-01-09,S03,DELIVERED
ORD005,C005,P004,3,800,2024-01-08,S03,DELIVERED
ORD006,C014,P008,3,9500,2024-01-24,S03,DELIVERED
ORD007,C032,P002,3,9500,2024-01-07,S01,DELIVERED
ORD008,C028,P004,2,1500,2024-01-14,S02,DELIVERED
ORD009,C011,P006,4,2000,2024-01-03,S03,DELIVERED
ORD010,C016,P007,3,50000,2024-01-05,S02,DELIVERED


In [0]:
orders_df.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: integer (nullable = true)
 |-- order_date: date (nullable = true)
 |-- store_id: string (nullable = true)
 |-- status: string (nullable = true)



In [0]:
print(f"Total Records : {orders_df.count()}")

Total Records : 20


In [0]:
orders_bronze_df = (
    orders_df
    .withColumn("ingestion_timestamp", current_timestamp())
    .withColumn("source_file", lit(batch_file))
)

In [0]:
display(orders_bronze_df)

order_id,customer_id,product_id,quantity,unit_price,order_date,store_id,status,ingestion_timestamp,source_file
ORD001,C026,P008,1,28000,2024-01-11,S02,DELIVERED,2026-07-11T17:14:27.923Z,orders_2024_01.csv
ORD002,C021,P004,4,18000,2024-01-12,S03,DELIVERED,2026-07-11T17:14:27.923Z,orders_2024_01.csv
ORD003,C020,P003,2,50000,2024-01-04,S01,DELIVERED,2026-07-11T17:14:27.923Z,orders_2024_01.csv
ORD004,C031,P007,1,18000,2024-01-09,S03,DELIVERED,2026-07-11T17:14:27.923Z,orders_2024_01.csv
ORD005,C005,P004,3,800,2024-01-08,S03,DELIVERED,2026-07-11T17:14:27.923Z,orders_2024_01.csv
ORD006,C014,P008,3,9500,2024-01-24,S03,DELIVERED,2026-07-11T17:14:27.923Z,orders_2024_01.csv
ORD007,C032,P002,3,9500,2024-01-07,S01,DELIVERED,2026-07-11T17:14:27.923Z,orders_2024_01.csv
ORD008,C028,P004,2,1500,2024-01-14,S02,DELIVERED,2026-07-11T17:14:27.923Z,orders_2024_01.csv
ORD009,C011,P006,4,2000,2024-01-03,S03,DELIVERED,2026-07-11T17:14:27.923Z,orders_2024_01.csv
ORD010,C016,P007,3,50000,2024-01-05,S02,DELIVERED,2026-07-11T17:14:27.923Z,orders_2024_01.csv


In [0]:
(
    orders_bronze_df.write
    .format("delta")
    .mode("overwrite")
    .save(f"{BRONZE_PATH}/bronze_orders")
)

In [0]:
bronze_orders = spark.read.format("delta").load(f"{BRONZE_PATH}/bronze_orders")

display(bronze_orders)

order_id,customer_id,product_id,quantity,unit_price,order_date,store_id,status,ingestion_timestamp,source_file
ORD001,C026,P008,1,28000,2024-01-11,S02,DELIVERED,2026-07-11T17:14:28.956Z,orders_2024_01.csv
ORD002,C021,P004,4,18000,2024-01-12,S03,DELIVERED,2026-07-11T17:14:28.956Z,orders_2024_01.csv
ORD003,C020,P003,2,50000,2024-01-04,S01,DELIVERED,2026-07-11T17:14:28.956Z,orders_2024_01.csv
ORD004,C031,P007,1,18000,2024-01-09,S03,DELIVERED,2026-07-11T17:14:28.956Z,orders_2024_01.csv
ORD005,C005,P004,3,800,2024-01-08,S03,DELIVERED,2026-07-11T17:14:28.956Z,orders_2024_01.csv
ORD006,C014,P008,3,9500,2024-01-24,S03,DELIVERED,2026-07-11T17:14:28.956Z,orders_2024_01.csv
ORD007,C032,P002,3,9500,2024-01-07,S01,DELIVERED,2026-07-11T17:14:28.956Z,orders_2024_01.csv
ORD008,C028,P004,2,1500,2024-01-14,S02,DELIVERED,2026-07-11T17:14:28.956Z,orders_2024_01.csv
ORD009,C011,P006,4,2000,2024-01-03,S03,DELIVERED,2026-07-11T17:14:28.956Z,orders_2024_01.csv
ORD010,C016,P007,3,50000,2024-01-05,S02,DELIVERED,2026-07-11T17:14:28.956Z,orders_2024_01.csv


In [0]:
# print("=" * 50)
print("Bronze Orders Layer Created Successfully")
print(f"Total Orders Loaded : {bronze_orders.count()}")
# print("=" * 50)

Bronze Orders Layer Created Successfully
Total Orders Loaded : 20


## Task 2

### Incremental Batch Load

In this step, the February order file is processed. Existing records are identified and removed before appending only the new orders to the Bronze layer.

In [0]:
incremental_df = read_csv(
    f"{DATA_PATH}/batch_incremental/orders_2024_02.csv"
)

In [0]:
display(incremental_df)

order_id,customer_id,product_id,quantity,unit_price,order_date,store_id,status
ORD003,C020,P003,2,50000,2024-01-04,S01,DELIVERED
ORD013,C029,P005,1,4000,2024-01-12,S02,DELIVERED
ORD021,C005,P003,3,1500,2024-02-09,S03,DELIVERED
ORD039,C023,P008,3,28000,2024-02-05,S04,DELIVERED
ORD040,C030,P002,4,18000,2024-02-15,S03,DELIVERED
ORD034,C013,P008,3,50000,2024-02-16,S02,DELIVERED
ORD026,C021,P003,4,18000,2024-02-06,S03,DELIVERED
ORD022,C010,P007,2,50000,2024-02-03,S01,DELIVERED
ORD025,C019,P007,1,1500,2024-02-03,S02,DELIVERED
ORD027,C003,P003,4,18000,2024-02-17,S04,DELIVERED


In [0]:
print(f"Incremental Records : {incremental_df.count()}")

Incremental Records : 22


In [0]:
bronze_orders = spark.read.format("delta").load(f"{BRONZE_PATH}/bronze_orders")

In [0]:
new_orders = (
    incremental_df.alias("inc")
    .join(
        bronze_orders.select("order_id").alias("br"),
        on="order_id",
        how="left_anti"
    )
)

In [0]:
display(new_orders)

order_id,customer_id,product_id,quantity,unit_price,order_date,store_id,status
ORD021,C005,P003,3,1500,2024-02-09,S03,DELIVERED
ORD039,C023,P008,3,28000,2024-02-05,S04,DELIVERED
ORD040,C030,P002,4,18000,2024-02-15,S03,DELIVERED
ORD034,C013,P008,3,50000,2024-02-16,S02,DELIVERED
ORD026,C021,P003,4,18000,2024-02-06,S03,DELIVERED
ORD022,C010,P007,2,50000,2024-02-03,S01,DELIVERED
ORD025,C019,P007,1,1500,2024-02-03,S02,DELIVERED
ORD027,C003,P003,4,18000,2024-02-17,S04,DELIVERED
ORD038,C016,P005,3,28000,2024-02-23,S04,DELIVERED
ORD036,C024,P008,1,9500,2024-02-06,S03,DELIVERED


In [0]:
print(f"New Orders : {new_orders.count()}")

New Orders : 20


In [0]:
new_orders = (
    new_orders
    .withColumn("ingestion_timestamp", current_timestamp())
    .withColumn("source_file", lit("orders_2024_02.csv"))
)

In [0]:
(
    new_orders.write
    .format("delta")
    .mode("append")
    .save(f"{BRONZE_PATH}/bronze_orders")
)

In [0]:
bronze_orders = spark.read.format("delta").load(f"{BRONZE_PATH}/bronze_orders")

display(bronze_orders)

order_id,customer_id,product_id,quantity,unit_price,order_date,store_id,status,ingestion_timestamp,source_file
ORD021,C005,P003,3,1500,2024-02-09,S03,DELIVERED,2026-07-11T17:14:37.724Z,orders_2024_02.csv
ORD039,C023,P008,3,28000,2024-02-05,S04,DELIVERED,2026-07-11T17:14:37.724Z,orders_2024_02.csv
ORD040,C030,P002,4,18000,2024-02-15,S03,DELIVERED,2026-07-11T17:14:37.724Z,orders_2024_02.csv
ORD034,C013,P008,3,50000,2024-02-16,S02,DELIVERED,2026-07-11T17:14:37.724Z,orders_2024_02.csv
ORD026,C021,P003,4,18000,2024-02-06,S03,DELIVERED,2026-07-11T17:14:37.724Z,orders_2024_02.csv
ORD022,C010,P007,2,50000,2024-02-03,S01,DELIVERED,2026-07-11T17:14:37.724Z,orders_2024_02.csv
ORD025,C019,P007,1,1500,2024-02-03,S02,DELIVERED,2026-07-11T17:14:37.724Z,orders_2024_02.csv
ORD027,C003,P003,4,18000,2024-02-17,S04,DELIVERED,2026-07-11T17:14:37.724Z,orders_2024_02.csv
ORD038,C016,P005,3,28000,2024-02-23,S04,DELIVERED,2026-07-11T17:14:37.724Z,orders_2024_02.csv
ORD036,C024,P008,1,9500,2024-02-06,S03,DELIVERED,2026-07-11T17:14:37.724Z,orders_2024_02.csv


In [0]:
print(f"Total Bronze Records : {bronze_orders.count()}")

Total Bronze Records : 40


In [0]:

print("Incremental Load Completed Successfully")
print(f"Final Bronze Records : {bronze_orders.count()}")


Incremental Load Completed Successfully
Final Bronze Records : 40


## Task 3

### Streaming Transaction Data using Auto Loader

In this step, transaction files are processed using Databricks Auto Loader and stored as a Delta table in the Bronze layer. Checkpoints are used to track processed files.

In [0]:
transactions_stream = (
    spark.readStream
         .format("cloudFiles")
         .option("cloudFiles.format", "csv")
         .option("header", "true")
         .option("inferSchema", "true")
         .option(
             "cloudFiles.schemaLocation",
             f"{CHECKPOINT_PATH}/schema/bronze_transactions"
         )
         .load(f"{DATA_PATH}/autoloader_landing")
)

In [0]:
transactions_stream.printSchema()

root
 |-- txn_id: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- amount: string (nullable = true)
 |-- txn_timestamp: string (nullable = true)
 |-- currency: string (nullable = true)
 |-- gateway_status: string (nullable = true)
 |-- _rescued_data: string (nullable = true)



In [0]:
stream_query = (
    transactions_stream.writeStream
        .format("delta")
        .option("checkpointLocation", f"{CHECKPOINT_PATH}/bronze_transactions")
        .option("path", f"{BRONZE_PATH}/bronze_transactions")
        .trigger(availableNow=True)
        .start()
)

In [0]:
stream_query.awaitTermination()

In [0]:
bronze_transactions = (
    spark.read
         .format("delta")
         .load(f"{BRONZE_PATH}/bronze_transactions")
)

In [0]:
display(bronze_transactions)

txn_id,order_id,payment_method,amount,txn_timestamp,currency,gateway_status,_rescued_data
TXN011,ORD035,DEBIT_CARD,19539.22,2024-03-01 03:00:00,INR,SUCCESS,null
TXN012,ORD028,CREDIT_CARD,37996.39,2024-03-02 11:00:00,INR,SUCCESS,null
TXN013,ORD002,CREDIT_CARD,38652.56,2024-03-01 22:00:00,INR,SUCCESS,null
TXN014,ORD_L02,CREDIT_CARD,30838.17,2024-03-01 04:00:00,INR,SUCCESS,null
TXN015,ORD022,NET_BANKING,11047.2,2024-03-02 05:00:00,INR,FAILED,null
TXN006,ORD024,UPI,23339.28,2024-03-01 13:00:00,INR,SUCCESS,null
TXN007,ORD016,UPI,25269.71,2024-03-02 01:00:00,INR,SUCCESS,null
TXN008,ORD021,DEBIT_CARD,26025.63,2024-03-02 22:00:00,INR,SUCCESS,null
TXN009,ORD030,DEBIT_CARD,18446.5,2024-03-02 08:00:00,INR,FAILED,null
TXN010,ORD035,DEBIT_CARD,48401.07,2024-03-01 05:00:00,INR,SUCCESS,null


In [0]:
print(f"Total Transactions : {bronze_transactions.count()}")

Total Transactions : 15


In [0]:
print("Streaming Load Completed Successfully")
print(f"Transactions Loaded : {bronze_transactions.count()}")


Streaming Load Completed Successfully
Transactions Loaded : 15


## Task 4

### Process Late Arriving Data

This task processes late-arriving January records using Delta Lake MERGE. Existing records are updated if required, while new records are inserted into the Bronze Orders table.

In [0]:
late_orders_df = read_csv(
    f"{DATA_PATH}/late_arriving/orders_2024_01_LATE.csv"
)

In [0]:
display(late_orders_df)

order_id,customer_id,product_id,quantity,unit_price,order_date,store_id,status
ORD_L01,C004,P003,4,4000,2024-01-15,S04,DELIVERED
ORD_L02,C002,P005,1,18000,2024-01-15,S04,DELIVERED
ORD_L03,C023,P005,4,2000,2024-01-19,S04,DELIVERED
ORD_L04,C017,P008,3,2000,2024-01-10,S04,DELIVERED
ORD_L05,C030,P001,1,4000,2024-01-09,S04,DELIVERED


In [0]:
late_orders_df = (
    late_orders_df
    .withColumn("ingestion_timestamp", current_timestamp())
    .withColumn("source_file", lit("orders_2024_01_LATE.csv"))
)

In [0]:
bronze_delta = DeltaTable.forPath(
    spark,
    f"{BRONZE_PATH}/bronze_orders"
)

In [0]:
(
    bronze_delta.alias("target")
    .merge(
        late_orders_df.alias("source"),
        "target.order_id = source.order_id"
    )
.whenMatchedUpdateAll()
.whenNotMatchedInsertAll()
.execute()
)

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [0]:
bronze_orders = spark.read.format("delta").load(f"{BRONZE_PATH}/bronze_orders")

display(bronze_orders)

order_id,customer_id,product_id,quantity,unit_price,order_date,store_id,status,ingestion_timestamp,source_file
ORD021,C005,P003,3,1500,2024-02-09,S03,DELIVERED,2026-07-11T17:14:37.724Z,orders_2024_02.csv
ORD039,C023,P008,3,28000,2024-02-05,S04,DELIVERED,2026-07-11T17:14:37.724Z,orders_2024_02.csv
ORD040,C030,P002,4,18000,2024-02-15,S03,DELIVERED,2026-07-11T17:14:37.724Z,orders_2024_02.csv
ORD034,C013,P008,3,50000,2024-02-16,S02,DELIVERED,2026-07-11T17:14:37.724Z,orders_2024_02.csv
ORD026,C021,P003,4,18000,2024-02-06,S03,DELIVERED,2026-07-11T17:14:37.724Z,orders_2024_02.csv
ORD022,C010,P007,2,50000,2024-02-03,S01,DELIVERED,2026-07-11T17:14:37.724Z,orders_2024_02.csv
ORD025,C019,P007,1,1500,2024-02-03,S02,DELIVERED,2026-07-11T17:14:37.724Z,orders_2024_02.csv
ORD027,C003,P003,4,18000,2024-02-17,S04,DELIVERED,2026-07-11T17:14:37.724Z,orders_2024_02.csv
ORD038,C016,P005,3,28000,2024-02-23,S04,DELIVERED,2026-07-11T17:14:37.724Z,orders_2024_02.csv
ORD036,C024,P008,1,9500,2024-02-06,S03,DELIVERED,2026-07-11T17:14:37.724Z,orders_2024_02.csv


In [0]:
s04_count = (
    bronze_orders
    .filter(col("store_id")=="S04")
    .count()
)

print("Late arriving records from S04 :", s04_count)

Late arriving records from S04 : 8


In [0]:
print(f"Total Bronze Records : {bronze_orders.count()}")

Total Bronze Records : 45


In [0]:

print("Late Arriving Data Processed Successfully")
print(f"Bronze Records : {bronze_orders.count()}")


Late Arriving Data Processed Successfully
Bronze Records : 45


# Silver Layer

The Silver layer cleans, enriches and joins business entities.

## Task 5

### Create Silver Layer

This step enriches the Bronze Orders table by joining customer, product, and store information. Business metrics such as revenue, cost, and margin are calculated before storing the result in the Silver layer.

In [0]:
customers_df = read_csv(
    f"{DATA_PATH}/customers.csv"
)

In [0]:
products_df = read_csv(
    f"{DATA_PATH}/products.csv"
)

In [0]:
stores_df = read_csv(
    f"{DATA_PATH}/stores.csv"
)

In [0]:
# Read Bronze Orders
bronze_orders = (
    spark.read
         .format("delta")
         .load(f"{BRONZE_PATH}/bronze_orders")
)

In [0]:
# Rename duplicate column before joining
stores_df = stores_df.withColumnRenamed("city", "store_city")

# Join all tables
silver_df = (
    bronze_orders
    .join(customers_df, on="customer_id", how="left")
    .join(products_df, on="product_id", how="left")
    .join(stores_df, on="store_id", how="left")
)

# Business Calculations
silver_df = (
    silver_df
    .withColumn("revenue", col("quantity") * col("unit_price"))
    .withColumn("total_cost", col("quantity") * col("cost_price"))
    .withColumn("margin", col("revenue") - col("total_cost"))
)

In [0]:
silver_df.printSchema()

root
 |-- store_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: integer (nullable = true)
 |-- order_date: date (nullable = true)
 |-- status: string (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)
 |-- source_file: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- tier: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- brand: string (nullable = true)
 |-- cost_price: integer (nullable = true)
 |-- store_name: string (nullable = true)
 |-- region: string (nullable = true)
 |-- store_city: string (nullable = true)
 |-- revenue: integer (nullable = true)
 |-- total_cost: integer (nullable = true)
 |-- margin: integer (nullable = true)



In [0]:
display(silver_df)

store_id,product_id,customer_id,order_id,quantity,unit_price,order_date,status,ingestion_timestamp,source_file,customer_name,city,tier,product_name,category,brand,cost_price,store_name,region,store_city,revenue,total_cost,margin
S03,P003,C005,ORD021,3,1500,2024-02-09,DELIVERED,2026-07-11T17:14:37.724Z,orders_2024_02.csv,Customer_5,Bangalore,GOLD,Headphones,Accessories,SoundMax,1500,Delhi NCR,North,Delhi,4500,4500,0
S04,P008,C023,ORD039,3,28000,2024-02-05,DELIVERED,2026-07-11T17:14:37.724Z,orders_2024_02.csv,Customer_23,Mumbai,BRONZE,Smartwatch,Accessories,TimeSmart,3000,Ahmedabad West,West,Ahmedabad,84000,9000,75000
S03,P002,C030,ORD040,4,18000,2024-02-15,DELIVERED,2026-07-11T17:14:37.724Z,orders_2024_02.csv,Customer_30,Bangalore,BRONZE,Smartphone,Electronics,CommX,25000,Delhi NCR,North,Delhi,72000,100000,-28000
S02,P008,C013,ORD034,3,50000,2024-02-16,DELIVERED,2026-07-11T17:14:37.724Z,orders_2024_02.csv,Customer_13,Mumbai,GOLD,Smartwatch,Accessories,TimeSmart,3000,Bangalore Hub,South,Bangalore,150000,9000,141000
S03,P003,C021,ORD026,4,18000,2024-02-06,DELIVERED,2026-07-11T17:14:37.724Z,orders_2024_02.csv,Customer_21,Bangalore,BRONZE,Headphones,Accessories,SoundMax,1500,Delhi NCR,North,Delhi,72000,6000,66000
S01,P007,C010,ORD022,2,50000,2024-02-03,DELIVERED,2026-07-11T17:14:37.724Z,orders_2024_02.csv,Customer_10,Ahmedabad,BRONZE,Tablet,Electronics,TabLite,15000,Mumbai Central,West,Mumbai,100000,30000,70000
S02,P007,C019,ORD025,1,1500,2024-02-03,DELIVERED,2026-07-11T17:14:37.724Z,orders_2024_02.csv,Customer_19,Bangalore,BRONZE,Tablet,Electronics,TabLite,15000,Bangalore Hub,South,Bangalore,1500,15000,-13500
S04,P003,C003,ORD027,4,18000,2024-02-17,DELIVERED,2026-07-11T17:14:37.724Z,orders_2024_02.csv,Customer_3,Delhi,BRONZE,Headphones,Accessories,SoundMax,1500,Ahmedabad West,West,Ahmedabad,72000,6000,66000
S04,P005,C016,ORD038,3,28000,2024-02-23,DELIVERED,2026-07-11T17:14:37.724Z,orders_2024_02.csv,Customer_16,Bangalore,GOLD,Keyboard,Accessories,TypeRight,1200,Ahmedabad West,West,Ahmedabad,84000,3600,80400
S03,P008,C024,ORD036,1,9500,2024-02-06,DELIVERED,2026-07-11T17:14:37.724Z,orders_2024_02.csv,Customer_24,Ahmedabad,GOLD,Smartwatch,Accessories,TimeSmart,3000,Delhi NCR,North,Delhi,9500,3000,6500


In [0]:
print("Null Value Validation")

display(
    silver_df.select(
        [
            sum(col(c).isNull().cast("int")).alias(c)
            for c in silver_df.columns
        ]
    )
)

print("Duplicate Order Validation")

print(
    silver_df.count()
    -
    silver_df.dropDuplicates(["order_id"]).count()
)

Null Value Validation


store_id,product_id,customer_id,order_id,quantity,unit_price,order_date,status,ingestion_timestamp,source_file,customer_name,city,tier,product_name,category,brand,cost_price,store_name,region,store_city,revenue,total_cost,margin
0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


Duplicate Order Validation
0


In [0]:
print("Row Count")

print(silver_df.count())

Row Count
45


In [0]:
(
    silver_df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .save(f"{SILVER_PATH}/silver_orders")
)

In [0]:
silver_orders = (
    spark.read
         .format("delta")
         .load(f"{SILVER_PATH}/silver_orders")
)

display(silver_orders)

print(f"Silver Records : {silver_orders.count()}")
print("Silver Layer Created Successfully")
print(f"Total Records : {silver_orders.count()}")

store_id,product_id,customer_id,order_id,quantity,unit_price,order_date,status,ingestion_timestamp,source_file,customer_name,city,tier,product_name,category,brand,cost_price,store_name,region,store_city,revenue,total_cost,margin
S02,P008,C026,ORD001,1,28000,2024-01-11,DELIVERED,2026-07-11T17:14:28.956Z,orders_2024_01.csv,Customer_26,Bangalore,SILVER,Smartwatch,Accessories,TimeSmart,3000,Bangalore Hub,South,Bangalore,28000,3000,25000
S03,P004,C021,ORD002,4,18000,2024-01-12,DELIVERED,2026-07-11T17:14:28.956Z,orders_2024_01.csv,Customer_21,Bangalore,BRONZE,Monitor,Electronics,VisionPlus,8000,Delhi NCR,North,Delhi,72000,32000,40000
S01,P003,C020,ORD003,2,50000,2024-01-04,DELIVERED,2026-07-11T17:14:28.956Z,orders_2024_01.csv,Customer_20,Delhi,BRONZE,Headphones,Accessories,SoundMax,1500,Mumbai Central,West,Mumbai,100000,3000,97000
S03,P007,C031,ORD004,1,18000,2024-01-09,DELIVERED,2026-07-11T17:14:28.956Z,orders_2024_01.csv,Customer_31,Delhi,GOLD,Tablet,Electronics,TabLite,15000,Delhi NCR,North,Delhi,18000,15000,3000
S03,P004,C005,ORD005,3,800,2024-01-08,DELIVERED,2026-07-11T17:14:28.956Z,orders_2024_01.csv,Customer_5,Bangalore,GOLD,Monitor,Electronics,VisionPlus,8000,Delhi NCR,North,Delhi,2400,24000,-21600
S03,P008,C014,ORD006,3,9500,2024-01-24,DELIVERED,2026-07-11T17:14:28.956Z,orders_2024_01.csv,Customer_14,Ahmedabad,BRONZE,Smartwatch,Accessories,TimeSmart,3000,Delhi NCR,North,Delhi,28500,9000,19500
S01,P002,C032,ORD007,3,9500,2024-01-07,DELIVERED,2026-07-11T17:14:28.956Z,orders_2024_01.csv,Customer_32,Pune,BRONZE,Smartphone,Electronics,CommX,25000,Mumbai Central,West,Mumbai,28500,75000,-46500
S02,P004,C028,ORD008,2,1500,2024-01-14,DELIVERED,2026-07-11T17:14:28.956Z,orders_2024_01.csv,Customer_28,Delhi,BRONZE,Monitor,Electronics,VisionPlus,8000,Bangalore Hub,South,Bangalore,3000,16000,-13000
S03,P006,C011,ORD009,4,2000,2024-01-03,DELIVERED,2026-07-11T17:14:28.956Z,orders_2024_01.csv,Customer_11,Bangalore,GOLD,Mouse,Accessories,ClickFast,600,Delhi NCR,North,Delhi,8000,2400,5600
S02,P007,C016,ORD010,3,50000,2024-01-05,DELIVERED,2026-07-11T17:14:28.956Z,orders_2024_01.csv,Customer_16,Bangalore,GOLD,Tablet,Electronics,TabLite,15000,Bangalore Hub,South,Bangalore,150000,45000,105000


Silver Records : 45
Silver Layer Created Successfully
Total Records : 45


# Gold Layer

The Gold layer contains analytics-ready business reports.

## Task 6

### Create Gold Layer

The Gold layer contains business-ready aggregated reports generated from the Silver layer. These reports support decision-making by summarizing sales and payment trends.

In [0]:
# Read Silver Layer
silver_orders = (
    spark.read
    .format("delta")
    .load(f"{SILVER_PATH}/silver_orders")
)

In [0]:
silver_orders.createOrReplaceTempView(
"silver_orders"
)

In [0]:
# Monthly Sales Summary
monthly_sales = spark.sql("""

SELECT

date_format(order_date,'yyyy-MM') as month,

COUNT(order_id) as total_orders,

ROUND(SUM(revenue),2) as total_sales,

ROUND(SUM(margin),2) as total_margin,

ROUND(AVG(revenue),2) as avg_order_value

FROM silver_orders

GROUP BY month

ORDER BY month

""")

In [0]:
display(monthly_sales)

month,total_orders,total_sales,total_margin,avg_order_value
2024-01,25,731900,71700,29276.0
2024-02,20,959000,615800,47950.0


In [0]:
(
    monthly_sales.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(f"{GOLD_PATH}/gold_monthly_sales")
)

In [0]:
# Sales by Category
sales_by_category = (
    silver_orders
    .groupBy("category")
    .agg(
        sum("revenue").alias("total_sales"),
        sum("margin").alias("total_margin"),
        sum("quantity").alias("total_quantity")
    )
)

In [0]:
display(sales_by_category)

category,total_sales,total_margin,total_quantity
Electronics,839900,-59100,53
Accessories,851000,746600,62


In [0]:
(
    sales_by_category.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema","true")
    .save(f"{GOLD_PATH}/gold_sales_by_category")
)

In [0]:
gold_monthly_sales = (
    spark.read
    .format("delta")
    .load(f"{GOLD_PATH}/gold_monthly_sales")
)

gold_sales_by_category = (
    spark.read
    .format("delta")
    .load(f"{GOLD_PATH}/gold_sales_by_category")
)

sales_by_region = (
    silver_orders
    .groupBy("region")
    .agg(
        sum("revenue").alias("total_sales"),
        sum("margin").alias("total_margin")
    )
)

display(sales_by_region)

(
sales_by_region.write
.format("delta")
.mode("overwrite")
.option("overwriteSchema","true")
.save(f"{GOLD_PATH}/gold_sales_by_region")
)

region,total_sales,total_margin
North,444900,122000
South,576000,240200
West,670000,325300


In [0]:
top_products = (
    silver_orders
    .groupBy("product_name")
    .agg(
        sum("revenue").alias("total_sales")
    )
    .orderBy(desc("total_sales"))
    .limit(5)
)
gold_top_products = (
    spark.read
    .format("delta")
    .load(f"{GOLD_PATH}/gold_top_products")
)
display(top_products)

(
    top_products.write
    .format("delta")
    .mode("overwrite")
    # .mode("overwrite")
.option("overwriteSchema","true")
    .save(f"{GOLD_PATH}/gold_top_products")
)
print(
f"Top Products : {gold_top_products.count()}"
)

product_name,total_sales
Tablet,369500
Headphones,355000
Smartwatch,308000
Smartphone,186000
Monitor,180400


Top Products : 5


In [0]:
display(gold_monthly_sales)

month,total_orders,total_sales,total_margin,avg_order_value
2024-01,25,731900,71700,29276.0
2024-02,20,959000,615800,47950.0


In [0]:
display(gold_sales_by_category)

category,total_sales,total_margin,total_quantity
Electronics,839900,-59100,53
Accessories,851000,746600,62


In [0]:
print("="*60)

print("RetailStream Pipeline Executed Successfully")

print("="*60)

print(f"Bronze Orders       : {bronze_orders.count()}")

print(f"Bronze Transactions : {bronze_transactions.count()}")

print(f"Silver Orders       : {silver_orders.count()}")

print(f"Gold Monthly Sales  : {gold_monthly_sales.count()}")

print(f"Gold Category Sales : {gold_sales_by_category.count()}")

print("="*60)

RetailStream Pipeline Executed Successfully
Bronze Orders       : 45
Bronze Transactions : 15
Silver Orders       : 45
Gold Monthly Sales  : 2
Gold Category Sales : 2


## Project Validation

- Bronze Layer Created Successfully

- Silver Layer Created Successfully

- Gold Layer Created Successfully

- Streaming Data Processed

- Incremental Load Successful

- Late Arriving Data Successfully Merged

In [0]:
assert gold_sales_by_category.count() > 0
assert top_products.count() > 0

## Conclusion

This project successfully implements an end-to-end RetailStream Data Engineering Pipeline using PySpark, Delta Lake and Databricks.

The pipeline processes historical batch data, incremental data, streaming transactions and late arriving records using a Medallion Architecture (Bronze, Silver and Gold).

The final Gold Layer produces analytics-ready datasets that can be directly connected to Power BI for business reporting and decision making.